### Open this notebook in Colab

Use the **GitHub → Colab** link with the exact org name **`Kr0issant`** — that is **K r 0 i s s a n t** (two letter **`s`** after the `i`). A single-`s` spelling (`Kr0isan`) breaks the URL and Colab shows **“Notebook not found”**.

**Working Colab URL (copy whole line):**

`https://colab.research.google.com/github/Kr0issant/communism-optimizer-rl/blob/main/notebooks/colab_grpo_training.ipynb`

If the repo was renamed on GitHub, try `nation-optimizer-rl` instead of `communism-optimizer-rl` in the same path.

# Nation Optimizer — GRPO training (OpenEnv + TRL)

This notebook reproduces the **TRL `GRPOTrainer`** + LoRA pipeline from [`training/train_grpo.py`](../training/train_grpo.py). The reward is **grounded in the same engine** as the OpenEnv server (`core` + `training/reward_fn.py`); prompts are loaded from the public Hub dataset (collected with [`scripts/collect_grpo_prompts.py`](../scripts/collect_grpo_prompts.py)).

**Hackathon bar:** OpenEnv (this repo’s `NationOpenEnv` / `openenv-core`), a **working TRL** training run, and **loss + reward** plots (saved under `--plot-path` or displayed below).

Runtime: **GPU** recommended (T4 or better). For a **CPU-only** sanity check, set `SMOKE = True` in the config cell (5 tiny steps, fp32).


## 1. Install `uv` and clone this repository

Dependencies are installed with **`uv`** to match the project [`README`](../README.md) (`uv sync --extra training`).

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

def sh(cmd: str) -> int:
    print(cmd, file=sys.stderr)
    return subprocess.call(cmd, shell=True)

if not Path(os.path.expanduser("~/.local/bin/uv")).exists():
    sh('curl -LsSf https://astral.sh/uv/install.sh | sh')
uv_bin = str(Path.home() / ".local/bin/uv")
os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ.get("PATH", "")
assert Path(uv_bin).exists(), f"uv not found at {uv_bin}"
print(subprocess.check_output([uv_bin, "--version"], text=True))

In [ ]:
REPO_URL = os.environ.get("NATION_OPT_REPO_URL", "https://github.com/Kr0issant/communism-optimizer-rl.git")
REPO_DIR = Path("/content/communism-optimizer-rl").resolve()
BRANCH = os.environ.get("NATION_OPT_REPO_BRANCH", "main")

if not REPO_DIR.is_dir():
    sh(f'git clone --depth 1 --branch {BRANCH} {REPO_URL!r} {REPO_DIR!s}')
else:
    sh(f'cd {REPO_DIR!s} && git pull')

os.chdir(REPO_DIR)
print("CWD:", os.getcwd())

In [ ]:
uv = str(Path.home() / ".local/bin/uv")
sh(f"cd {REPO_DIR!s} && {uv!s} sync --extra training")
print("Ready.")

## 2. (Optional) Hugging Face token

The **prompt dataset** and **base model** are public. A token is only needed if you set `--push` or hit Hub rate limits. In Colab you can use a secret `HF_TOKEN` or run an interactive login.

In [ ]:
import os
if "HF_TOKEN" in os.environ and os.environ["HF_TOKEN"]:
    from huggingface_hub import login
    login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
    print("HF login: ok (env)")
else:
    print("Set HF_TOKEN or run: from huggingface_hub import login; login()")

## 3. OpenEnv smoke: `NationOpenEnv`

Confirms the same **OpenEnv** wrapper used for the [Hugging Face Space](https://huggingface.co/spaces) entry loads and steps once.

In [ ]:
uv = str(Path.home() / ".local/bin/uv")
code = r"""
from server.environment import NationOpenEnv
from server.models import NationAction
env = NationOpenEnv()
obs = env.reset(seed=42, episode_id="colab-openenv-smoke")
r = env.step(NationAction(actions=[]))
print("phase", r.state.get("phase"), "round", r.state.get("round"), "reward", r.reward)
print("OpenEnv smoke OK.")
"""
sh(f"cd {REPO_DIR!s} && {uv!s} run python -c {code!r}")

## 4. GRPO + LoRA training (TRL)

Runs `uv run python training/train_grpo.py` with the same defaults as the repo. Set `SMOKE = True` for a fast **CPU** check; set `False` and use a **GPU** for a real short run. Plots are written to `OUT_PLOT` and shown inline.

In [ ]:
import os
SMOKE = True  # False for a longer GPU run (uncheck CPU runtime)
_u = os.environ.get("NATION_HF_USER", "abanwild")
DATASET_ID = os.environ.get("NATION_DATASET_ID", f"{_u}/nation-parliamentary-prompts")
OUT_PLOT = str(REPO_DIR / "out/colab_train_curves.png")
EXTRA_ARGS: list[str] = []

uv = str(Path.home() / ".local/bin/uv")
args = [uv, "run", "python", "training/train_grpo.py", "--no-push", "--report-to", "none"]
args += ["--dataset-id", DATASET_ID, "--plot-path", OUT_PLOT]
if SMOKE:
    args.append("--smoke")
else:
    # Short Colab-friendly defaults (adjust for your budget)
    args += ["--max-steps", "50", "--per-device-batch-size", "1", "--num-generations", "2"]
args += EXTRA_ARGS
print(" ".join(args))
rc = subprocess.call(args, cwd=REPO_DIR)
if rc != 0:
    raise RuntimeError(f"training/train_grpo.py exited with {rc}")

In [ ]:
from IPython.display import Image, display
from pathlib import Path as P
p = P(OUT_PLOT)
if p.is_file():
    display(Image(filename=str(p)))
else:
    print("No plot at", p, "(training may have failed or SMOKE produced no plottable logs).")

## 5. Next steps

- Full runs and Hub push: see [`README` ../README.md#reproduce-the-run](../README.md#reproduce-the-run) and [`training/train_grpo.py`](../training/train_grpo.py).
- **Trackio** (optional): pass `--report-to trackio` when training, or use `--plot-path` for local plots.
- Regenerate the prompt dataset: `uv run python -m scripts.collect_grpo_prompts`.
